# Build drug resources

**Purpose.** This support notebook builds, *once*, two drug resources with different units of observation
that the main pipeline then only *reads*:

- `data/external/aifa_products.csv` - one row per AIFA product (the 14,976 *anagrafica farmaci* product
  names). It supplies product-name lookup for the Step-2 exact gazetteer and the product-label retrieval
  index used by the Step-5 ATC fallback, and it carries product-level ATC metadata.
- `data/external/ingredient_crosswalk.csv` - one row per canonical single-substance AIFA active
  ingredient, aligning that ingredient with its ATC, a candidate RxNorm RXCUI and a candidate DDInter id,
  each with provenance. It is resolved over the **full AIFA registry**, not over this cohort, so a drug
  absent from these patients is still covered.

Product and ingredient are different levels of granularity, so the two files are deliberately separate.
The crosswalk is not a universal ontology: `RXCUI` and `DDINTER` are candidate links with a `*_source`
(and, for DDInter, a `ddinter_similarity`), not certified equivalences. The Step-2 extractor is symbolic
and exact; the product-label e5 embeddings are used only by the downstream Step-5 ATC fallback, not by the
operational Step-2 extraction. The main pipeline does not recompute the RxNorm or DDInter joins.

| column (`ingredient_crosswalk`) | meaning | source & method |
|---|---|---|
| `atc_exact` / `atc_link` | representative monotherapy ATC (the most frequent code in the AIFA package table) | **AIFA** package table (single-substance products), salt-stripped |
| `atc4` | ATC level-4 (first 5 chars of `atc_link`, only when the code is at least 5 chars long) | derived |
| `RXCUI` | RxNorm ingredient CUI, a **candidate link** | **RxNav** approximate-match endpoint, verified against the AIFA ATC-4 |
| `DDINTER` | DDInter drug id, a **candidate link** | **DDInter** drug list, dense-retrieval name match >= 0.90 |
| `*_source`, `ddinter_similarity` | provenance / confidence of each code | derived |

**Why offline.** The RxNorm and DDInter codes are *not* present in the raw AIFA input (those columns ship
empty). They are resolved once here - RxCUI from a one-off RxNav crawl saved to
`data/external/rx/rxnorm/rxnorm_ingredients_aifa.csv`, DDInter by e5 retrieval - yielding populated,
inspectable, version-controlled artifacts with explicit provenance, so the main pipeline simply *loads* them.

**Input.** the pre-downloaded resources under `data/external/`: the raw AIFA dictionary
`data/external/aifa/normalized/medications.csv`, the AIFA package table
`data/external/aifa/normalized/aifa_confezioni.parquet`, and the pre-crawled RxNav ingredient table
`data/external/rx/rxnorm/rxnorm_ingredients_aifa.csv`.
**Output.** `data/external/aifa_products.csv` and `data/external/ingredient_crosswalk.csv`.

In [1]:
import re, json, hashlib
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
EXTERNAL = ROOT / "data" / "external"
OUTPUTS  = ROOT / "outputs"
RAW_AIFA = EXTERNAL / "aifa" / "normalized" / "medications.csv"   # raw AIFA anagrafica (builder input)
PRODUCTS_DEST  = EXTERNAL / "aifa_products.csv"                    # product-level resource (pipeline input)
CROSSWALK_DEST = EXTERNAL / "ingredient_crosswalk.csv"             # ingredient-level crosswalk (pipeline input)
EXTERNAL.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)                         # the embedding cache is written here
PRODUCTS_DEST.parent.mkdir(parents=True, exist_ok=True)
CROSSWALK_DEST.parent.mkdir(parents=True, exist_ok=True)
# quiet the transformers/HF progress bars so the saved output stays clean
from transformers.utils import logging as _hf_logging
_hf_logging.set_verbosity_error(); _hf_logging.disable_progress_bar()
try:
    from huggingface_hub.utils import disable_progress_bars as _dpb; _dpb()
except Exception:
    pass

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

device: cuda


In [2]:
# ---- AIFA monotherapy name -> ATC (single-ingredient packages, most frequent ATC, salt-stripped) ----
raw = pd.read_csv(RAW_AIFA)
assert raw.ATC.notna().all(), "raw AIFA ATC has NaN -> would break the e5-cache alignment"
aifa = raw[["preferred_label", "ATC"]].copy()
aifa["atc1"] = aifa.ATC.str.split("|").str[0]
DICT_LABELS = aifa.preferred_label.astype(str).tolist()

conf = pd.read_parquet(EXTERNAL / "aifa" / "normalized" / "aifa_confezioni.parquet")
_mono = conf[~conf.PA_ASSOCIATI.astype(str).str.contains("/", na=False)].dropna(subset=["PA_ASSOCIATI", "CODICE_ATC"])
name2atc = (_mono.groupby(_mono.PA_ASSOCIATI.str.upper().str.strip())["CODICE_ATC"]
            .agg(lambda s: s.value_counts().index[0]).to_dict())

_SALT_RE = re.compile(r'\b(cloridrato|idrocloruro|solfato|sodico|sodica|calcio|calcico|besilato|'
                      r'fumarato|emifumarato|maleato|mesilato|potassico|bromuro|acetato|tartrato|'
                      r'succinato|dipropionato|furoato|valerato|di sodio|pentaidrato|monoidrato|'
                      r'biidrato|triidrato|anidro|nitrato|fosfato|citrato)\b', re.I)
def _strip_salt(n): return re.sub(r'\s+', ' ', _SALT_RE.sub('', n)).strip()
for _k, _v in list(name2atc.items()):
    _b = _strip_salt(_k)
    if _b and _b != _k:
        name2atc.setdefault(_b, _v)

def link_atc(name):
    u = re.sub(r'\s*\d.*$', '', str(name).upper()).strip()
    if not u:
        return None
    return name2atc.get(u) or name2atc.get(_strip_salt(u))

# Canonical drug key (shared verbatim with the main pipeline): strip a trailing dose/instruction tail
# and salt, lowercase. Collapses "Atorvastatina 40 mg" / "atorvastatina" -> "atorvastatina", while a
# pure dose fragment ("10 mg 1 cp alle ore 22") reduces to a non-drug string that is discarded later.
_DOSE_TAIL = re.compile(r'\b(?:mg|mcg|ml|ui|cp|cpr|cps|gtt|fl|bustin[ae]|bust|sciroppo|scir|soluz|'
                        r'gocce|nipio|fiale|fiala|die|ore|da assumere)\b.*$', re.I)
def normalize_drug_key(value):
    v = re.sub(r'\s+', ' ', str(value).strip())
    v = _DOSE_TAIL.sub('', v)
    v = re.sub(r'\s+\d+(?:[.,]\d+)?\s*$', '', v)      # trailing bare dose number
    # keep the original label if salt/dose reduction empties the key (e.g. "Calcio", "Calcio acetato")
    return (_strip_salt(v.upper()).strip() or str(value).strip().upper()).lower().strip()

aifa["atc_link"] = [link_atc(l) or a for l, a in zip(aifa.preferred_label, aifa.atc1)]
print(f"AIFA dictionary entries: {len(aifa)} | monotherapy name->ATC keys: {len(name2atc)}")

AIFA dictionary entries: 14976 | monotherapy name->ATC keys: 2845


In [3]:
# ---- e5 embedder + dense-retrieval over the AIFA dictionary (for the ATC fallback) ----
# The cached embeddings are guarded by a hash of the labels + model id, so a stale or re-ordered
# dictionary can never be silently paired with the wrong embeddings.
embedder = SentenceTransformer("intfloat/multilingual-e5-small", device=DEVICE)
MODEL_ID = "intfloat/multilingual-e5-small"
_cache = OUTPUTS / "aifa_e5_emb.npy"
_meta  = OUTPUTS / "aifa_e5_emb.meta.json"
_labels_hash = hashlib.sha256("\n".join(DICT_LABELS).encode("utf-8")).hexdigest()
_valid = False
if _cache.exists() and _meta.exists():
    m = json.loads(_meta.read_text(encoding="utf-8"))
    _valid = (m.get("model") == MODEL_ID and m.get("rows") == len(DICT_LABELS)
              and m.get("labels_sha256") == _labels_hash)
if _valid:
    dict_emb = torch.tensor(np.load(_cache), device=DEVICE)
else:
    dict_emb = embedder.encode(["passage: " + l for l in DICT_LABELS], batch_size=256,
                               convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
    np.save(_cache, dict_emb.cpu().numpy())
    _meta.write_text(json.dumps({"model": MODEL_ID, "rows": len(DICT_LABELS),
                                 "labels_sha256": _labels_hash}), encoding="utf-8")
assert dict_emb.shape[0] == len(DICT_LABELS)

def retrieve(queries, topk=1):
    if not queries:
        return []
    q = embedder.encode(["query: " + s for s in queries], convert_to_tensor=True,
                        normalize_embeddings=True, show_progress_bar=False)
    vals, idx = torch.topk(q @ dict_emb.T, topk, dim=1)
    return [[(float(vals[i, k]), DICT_LABELS[idx[i, k]], aifa.iloc[int(idx[i, k])].atc_link)
             for k in range(topk)] for i in range(len(queries))]
print("AIFA retrieval index ready:", tuple(dict_emb.shape), "| cache valid:", _valid)

AIFA retrieval index ready: (14976, 384) | cache valid: True


In [4]:
# ---- AIFA-wide active ingredients: every single-substance ingredient in the AIFA registry, with its
# most-frequent ATC. This code table is derived from AIFA itself (NOT from the patient cohort), so the
# alert engine can resolve any AIFA drug, not only the ones these patients happen to take.
_mono = conf[~conf.PA_ASSOCIATI.astype(str).str.contains("/", na=False)].dropna(subset=["PA_ASSOCIATI", "CODICE_ATC"]).copy()
_mono["ing"] = _mono.PA_ASSOCIATI.str.strip().str.title()
ing2atc = _mono.groupby("ing")["CODICE_ATC"].agg(lambda s: s.value_counts().index[0]).to_dict()
ingredients_df = pd.DataFrame({"ingredient": list(ing2atc), "atc_link": list(ing2atc.values())})
ingredients_df["lookup_key"] = ingredients_df.ingredient.map(normalize_drug_key)
ingredients_df = (ingredients_df.dropna(subset=["atc_link"])
                  .drop_duplicates("lookup_key").sort_values("lookup_key").reset_index(drop=True))
print(f"AIFA single-substance ingredients: {len(ingredients_df)}")

AIFA single-substance ingredients: 2343


In [5]:
# ---- RxCUI per ingredient from the pre-crawled, ATC-4-verified RxNav table.
# rxnorm_ingredients_aifa.csv is built once by the RxNav crawl (approximate-match endpoint, verified
# against the AIFA ATC-4; see README). Joining it here keeps this build fully offline.
_rxa = pd.read_csv(EXTERNAL / "rx" / "rxnorm" / "rxnorm_ingredients_aifa.csv")
_rxa["lookup_key"] = _rxa.ingredient.map(normalize_drug_key)
rx_by_key = {k: int(v) for k, v in zip(_rxa.lookup_key, _rxa.rxcui) if pd.notna(v)}
ingredients_df["RXCUI"] = ingredients_df.lookup_key.map(rx_by_key)
print(f"RxCUI resolved: {ingredients_df.RXCUI.notna().sum()}/{len(ingredients_df)} "
      f"({100 * ingredients_df.RXCUI.notna().mean():.0f}%, RxNav approximate match, ATC-4 verified)")

RxCUI resolved: 1584/2343 (68%, RxNav approximate match, ATC-4 verified)


In [6]:
# ---- DDInter id per ingredient: dense-retrieval match (>= 0.90) of the ingredient name against the
# DDInter drug list, similarity kept as provenance.
ddd = pd.read_csv(EXTERNAL / "ddinter" / "normalized" / "ddinter_drugs.csv")
_dd_emb = embedder.encode(["passage: " + x for x in ddd.preferred_label.astype(str)], batch_size=256,
                          convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
_qi = embedder.encode(["query: " + x for x in ingredients_df.ingredient], batch_size=256,
                      convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
_vi, _ii = torch.topk(_qi @ _dd_emb.T, 1, dim=1)
DDINTER_SIM = 0.90
ingredients_df["DDINTER"] = [ddd.iloc[int(_ii[k, 0])].DDINTER if float(_vi[k, 0]) >= DDINTER_SIM else None
                             for k in range(len(ingredients_df))]
ingredients_df["ddinter_similarity"] = [round(float(_vi[k, 0]), 6) for k in range(len(ingredients_df))]
print(f"DDInter id resolved (>= {DDINTER_SIM}): {ingredients_df.DDINTER.notna().sum()}/{len(ingredients_df)}")

DDInter id resolved (>= 0.9): 1242/2343


In [7]:
# ---- AIFA product layer: every AIFA product name, used as the exact-match symbolic ingress gazetteer
# (Step 2) AND as the dense-retrieval fallback for discharge ATC (Step 5). Only the label and its ATC are needed
# here; the alert-engine codes live in the ingredient
# crosswalk built above.
med = raw.copy()
med["normalized_label"] = med.preferred_label.map(normalize_drug_key)
med["atc_monotherapy"]  = med.preferred_label.map(link_atc)
med["atc_link"]         = med.atc_monotherapy.fillna(med.ATC.astype(str).str.split("|").str[0])
med["atc4"]             = med.atc_link.where(med.atc_link.astype("string").str.len().ge(5)).str[:5]
med["atc_exact"]        = med.atc_monotherapy
med["atc_source"]       = np.where(med.atc_monotherapy.notna(), "monotherapy_confezioni", "aifa_first_atc")
print(f"AIFA product retrieval rows: {len(med)}")

AIFA product retrieval rows: 14976


In [8]:
# ---- Product-level resource and ingredient-level crosswalk (two independent files) ----------------
PRODUCT_COLUMNS = ["preferred_label", "entity_type", "source", "aliases", "AIC", "ATC",
                   "normalized_label", "atc_monotherapy", "atc_link", "atc4", "atc_exact", "atc_source"]
CROSSWALK_COLUMNS = ["preferred_label", "entity_type", "source", "aliases", "lookup_key",
                     "atc_link", "atc4", "atc_exact", "atc_source",
                     "RXCUI", "rxcui_source", "DDINTER", "ddinter_source", "ddinter_similarity"]

# Product-level resource: one row per AIFA product (catalog + product-label ATC retrieval metadata).
aifa_products = med.copy()
aifa_products["entity_type"] = aifa_products.get("entity_type", "DRUG")
aifa_products["source"] = "AIFA anagrafica farmaci"
aifa_products["AIC"] = aifa_products["AIC"].astype("string")

# Ingredient-level crosswalk: one canonical single-substance active ingredient per row, aligned to ATC,
# a candidate RxNorm RXCUI and a candidate DDInter id, each with provenance.
ingredient_crosswalk = pd.DataFrame({
    "preferred_label": ingredients_df.ingredient,
    "entity_type": "INGREDIENT",
    "source": "AIFA package table",
    "aliases": ingredients_df.ingredient,
    "lookup_key": ingredients_df.lookup_key,
    "atc_link": ingredients_df.atc_link,
    "atc4": ingredients_df.atc_link.where(ingredients_df.atc_link.astype("string").str.len().ge(5)).str[:5],
    "atc_exact": ingredients_df.atc_link,
    "atc_source": "monotherapy_confezioni",
    "RXCUI": ingredients_df.RXCUI,
    "rxcui_source": np.where(ingredients_df.RXCUI.notna(), "rxnav_atc_verified", "not_found"),
    "DDINTER": ingredients_df.DDINTER,
    "ddinter_source": np.where(ingredients_df.DDINTER.notna(), "ddinter_retrieval", "not_found"),
    "ddinter_similarity": ingredients_df.ddinter_similarity,
})

# explicit dtypes
ingredient_crosswalk["lookup_key"] = ingredient_crosswalk["lookup_key"].astype("string")
ingredient_crosswalk["RXCUI"] = pd.to_numeric(ingredient_crosswalk["RXCUI"], errors="coerce").astype("Int64")
ingredient_crosswalk["DDINTER"] = ingredient_crosswalk["DDINTER"].astype("string")
ingredient_crosswalk["ddinter_similarity"] = pd.to_numeric(ingredient_crosswalk["ddinter_similarity"], errors="coerce")

# ---- validations ----
assert len(aifa_products) == len(raw)
assert aifa_products["preferred_label"].notna().all() and aifa_products["preferred_label"].str.strip().ne("").all()
assert aifa_products["normalized_label"].notna().all() and aifa_products["normalized_label"].str.strip().ne("").all()
assert aifa_products["atc_link"].notna().all()
assert aifa_products["atc4"].dropna().str.len().eq(5).all()

assert len(ingredient_crosswalk) == len(ingredients_df)
assert ingredient_crosswalk["preferred_label"].notna().all() and ingredient_crosswalk["preferred_label"].str.strip().ne("").all()
assert ingredient_crosswalk["lookup_key"].notna().all() and ingredient_crosswalk["lookup_key"].str.strip().ne("").all()
assert ingredient_crosswalk["lookup_key"].is_unique
assert ingredient_crosswalk["atc_link"].notna().all()
assert ingredient_crosswalk["atc4"].dropna().str.len().eq(5).all()
# regression guard: a salt/dose-only ingredient label keeps a non-empty canonical key ("Calcio" -> "calcio")
assert ingredient_crosswalk.loc[ingredient_crosswalk["preferred_label"].str.casefold().eq("calcio"), "lookup_key"].eq("calcio").all()
# every accepted DDInter link satisfies the threshold on the unrounded decision value
assert ingredient_crosswalk.loc[ingredient_crosswalk["DDINTER"].notna(), "ddinter_similarity"].ge(DDINTER_SIM).all()

aifa_products[PRODUCT_COLUMNS].to_csv(PRODUCTS_DEST, index=False)
ingredient_crosswalk[CROSSWALK_COLUMNS].to_csv(CROSSWALK_DEST, index=False)

print("wrote -> data/external/aifa_products.csv")
print(f"rows: {len(aifa_products)}")
print()
print("wrote -> data/external/ingredient_crosswalk.csv")
print(f"rows: {len(ingredient_crosswalk)}")
print(f"ATC: {int(ingredient_crosswalk.atc_link.notna().sum())}")
print(f"RxCUI: {int(ingredient_crosswalk.RXCUI.notna().sum())}")
print(f"DDInter: {int(ingredient_crosswalk.DDINTER.notna().sum())}")
display(aifa_products[PRODUCT_COLUMNS].head(5))
display(ingredient_crosswalk[CROSSWALK_COLUMNS].head(5))

wrote -> data/external/aifa_products.csv
rows: 14976

wrote -> data/external/ingredient_crosswalk.csv
rows: 2343
ATC: 2343
RxCUI: 1584
DDInter: 1242


,preferred_label,entity_type,source,aliases,AIC,ATC,normalized_label,atc_monotherapy,atc_link,atc4,atc_exact,atc_source
0,131-I IODOMETHYL NORCHOLESTEROL,DRUG,AIFA anagrafica farmaci,131-I IODOMETHYL NORCHOLESTEROL,039028016,V09XA01,131-i iodomethyl norcholesterol,None,V09XA01,V09XA,None,aifa_first_atc
1,ABACAVIR,DRUG,AIFA anagrafica farmaci,ABACAVIR,043618014|043618026|043618038|045348012|045348...,J05AF06|J05AR02|J05AR13,abacavir,J05AF06,J05AF06,J05AF,J05AF06,monotherapy_confezioni
2,ABACAVIR E LAMIVUDINA AUROBINDO,DRUG,AIFA anagrafica farmaci,ABACAVIR E LAMIVUDINA AUROBINDO,044669012|044669024|044669036|044669048|044669...,J05AR02,abacavir e lamivudina aurobindo,None,J05AR02,J05AR,None,aifa_first_atc
3,ABACAVIR E LAMIVUDINA SUN,DRUG,AIFA anagrafica farmaci,ABACAVIR E LAMIVUDINA SUN,045348012|045348036|045348051|045348063,J05AR02,abacavir e lamivudina sun,None,J05AR02,J05AR,None,aifa_first_atc
4,ABACAVIR MYLAN,DRUG,AIFA anagrafica farmaci,ABACAVIR MYLAN,045354014|045354026|045354038|045354040,J05AF06,abacavir mylan,None,J05AF06,J05AF,None,aifa_first_atc


,preferred_label,entity_type,source,aliases,lookup_key,atc_link,atc4,atc_exact,atc_source,RXCUI,rxcui_source,DDINTER,ddinter_source,ddinter_similarity
0,131-I Iodomethyl Norcholesterol,INGREDIENT,AIFA package table,131-I Iodomethyl Norcholesterol,131-i iodomethyl norcholesterol,V09XA01,V09XA,V09XA01,monotherapy_confezioni,<NA>,not_found,<NA>,not_found,0.883283
1,[18F]Psma-1007,INGREDIENT,AIFA package table,[18F]Psma-1007,[18f]psma-1007,V09IX,V09IX,V09IX,monotherapy_confezioni,2586016,rxnav_atc_verified,<NA>,not_found,0.847920
2,Abacavir,INGREDIENT,AIFA package table,Abacavir,abacavir,J05AF06,J05AF,J05AF06,monotherapy_confezioni,190521,rxnav_atc_verified,DDInter1,ddinter_retrieval,0.924199
3,Abaloparatide,INGREDIENT,AIFA package table,Abaloparatide,abaloparatide,H05AA04,H05AA,H05AA04,monotherapy_confezioni,1921069,rxnav_atc_verified,DDInter2,ddinter_retrieval,0.944689
4,Abatacept,INGREDIENT,AIFA package table,Abatacept,abatacept,L04AA24,L04AA,L04AA24,monotherapy_confezioni,614391,rxnav_atc_verified,DDInter5,ddinter_retrieval,0.935563


**How the main pipeline consumes this.** `clinical_nlp_pipeline.ipynb` loads the two resources
separately:

- `aifa_products.csv` supplies the AIFA product labels for the Step-2 exact gazetteer and the
  product-label dense-retrieval index used as the discharge-therapy ATC fallback (Step 5);
- `ingredient_crosswalk.csv` supplies the codes the alert engine reads: **Step 5** maps each discharge
  active ingredient (via the *same* `normalize_drug_key`) to its `atc_link` / `atc4`, with a
  dense-retrieval ATC fallback applied in the main pipeline for valid ingredient names absent from the
  crosswalk, and **Step 6** reads `RXCUI` and `DDINTER`.

The condition-to-disease mapping used by the drug-disease contraindication rule stays in the main
pipeline, because it depends on the *observed patient conditions*, not on a static drug resource. The
`RXCUI` and `DDINTER` values are **candidate links** (with a `*_source` and, for DDInter, a
`ddinter_similarity`), not certified equivalences.